In [8]:
%pip -q install geemap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 23.3 MB/s eta 0:00:00


In [9]:
import ee
import geemap

# Authenticate and initialize
try:
    ee.Initialize(project='ee-natthanichap')
except:
    ee.Authenticate()
    ee.Initialize(project='ee-natthanichap')

# Study Area
roi = ee.FeatureCollection("FAO/GAUL/2015/level1") \
    .filter(ee.Filter.eq('ADM1_NAME', 'Chiang Mai'))

# Fuel
lc = ee.Image("MODIS/061/MCD12Q1/2022_01_01") \
    .select('LC_Type1') \
    .clip(roi)

fuel = lc.remap(
    [1,2,3,4,5,8,9,10,12,13,17],
    [1,1,1,1,1,0.6,0.6,1,0.3,0.1,0],
    0
).toFloat().rename('fuel')

# NDVI
ndvi = ee.ImageCollection("MODIS/061/MOD13Q1") \
    .filterDate('2023-01-01', '2023-05-30') \
    .select('NDVI') \
    .median() \
    .clip(roi)

dryness = ndvi.multiply(0.0001) \
    .unitScale(0, 0.6) \
    .multiply(-1).add(1) \
    .clamp(0, 1) \
    .rename('dryness')

# Slope
dem = ee.Image("USGS/SRTMGL1_003").clip(roi)
slope = ee.Terrain.slope(dem)

slopeNorm = slope.unitScale(0, 30) \
    .clamp(0, 1) \
    .rename('slope')

# Model
risk = fuel.multiply(0.40) \
    .add(dryness.multiply(0.30)) \
    .add(slopeNorm.multiply(0.10)) \
    .rename('Risk')

# Stats
stats = risk.reduceRegion(
    reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), '', True),
    geometry=roi.geometry(),
    scale=1000,
    maxPixels=1e9
)

print(stats.getInfo())

{'Risk_mean': 0.3336236578218496, 'Risk_stdDev': 0.10832039757187845}


In [10]:
import ee
import geemap

# -----------------------------
# 0. Study Area
# -----------------------------
roi = ee.FeatureCollection("FAO/GAUL/2015/level1") \
    .filter(ee.Filter.eq('ADM1_NAME', 'Chiang Mai'))

# -----------------------------
# 1. FACTORS
# -----------------------------

# Factor 1: Fuel availability from land cover
lc = ee.Image("MODIS/061/MCD12Q1/2022_01_01") \
    .select('LC_Type1') \
    .clip(roi)

fuel = lc.remap(
    [1, 2, 3, 4, 5, 8, 9, 10, 12, 13, 17],
    [1, 1, 1, 1, 1, 0.6, 0.6, 1, 0.3, 0.1, 0],
    0
).toFloat().rename('fuel')

# Factor 2: Vegetation dryness proxy using NDVI
ndvi = ee.ImageCollection("MODIS/061/MOD13Q1") \
    .filterDate('2023-01-01', '2023-05-30') \
    .select('NDVI') \
    .median() \
    .clip(roi)

dryness = ndvi.multiply(0.0001) \
    .unitScale(0, 0.6) \
    .multiply(-1).add(1) \
    .clamp(0, 1) \
    .rename('dryness')

# Factor 3: Proximity to urban areas (human ignition proxy)
urban = lc.eq(13).selfMask()

distUrban = urban.distance(ee.Kernel.euclidean(5000, 'meters')) \
    .clip(roi)

human = distUrban.unitScale(0, 5000) \
    .multiply(-1).add(1) \
    .clamp(0, 1) \
    .rename('human')

# Factor 4: Slope from DEM
dem = ee.Image("USGS/SRTMGL1_003").clip(roi)
slope = ee.Terrain.slope(dem)

slopeNorm = slope.unitScale(0, 30) \
    .clamp(0, 1) \
    .rename('slope')

# -----------------------------
# 2. WEIGHTED LINEAR COMBINATION
# -----------------------------
risk = fuel.multiply(0.40) \
    .add(dryness.multiply(0.30)) \
    .add(human.multiply(0.20)) \
    .add(slopeNorm.multiply(0.10)) \
    .rename('Risk')

# -----------------------------
# 3. CLASSIFICATION
# Dynamic threshold using mean +- 0.5 std
# -----------------------------
stats = risk.reduceRegion(
    reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs=True),
    geometry=roi.geometry(),
    scale=500,
    maxPixels=1e9
)

mean = ee.Number(stats.get('Risk_mean'))
std = ee.Number(stats.get('Risk_stdDev'))

riskClass = risk.expression(
    "(r < mean - 0.5 * std) ? 1"
    ": (r > mean + 0.5 * std) ? 3"
    ": 2",
    {
        'r': risk,
        'mean': mean,
        'std': std
    }
).rename('Risk_Class')

# -----------------------------
# 4. VALIDATION WITH REAL DATA
# -----------------------------
burned = ee.ImageCollection("MODIS/061/MCD64A1") \
    .filterDate('2023-01-01', '2023-05-30') \
    .select('BurnDate') \
    .max() \
    .clip(roi)

fireMask = burned.gt(0).rename('burned')

highRisk = riskClass.eq(3).rename('highRisk')
agreement = highRisk.And(fireMask).selfMask()

validationStats = ee.Image.cat([
    highRisk.rename('highRisk'),
    fireMask.rename('burned'),
    agreement.rename('agreement')
]).reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=roi.geometry(),
    scale=500,
    maxPixels=1e9
)

# -----------------------------
# 5. SENSITIVITY ANALYSIS
# Adjust most important factor (Fuel) by +-20%
# -----------------------------
risk_fuel_plus = fuel.multiply(0.48) \
    .add(dryness.multiply(0.26)) \
    .add(human.multiply(0.17)) \
    .add(slopeNorm.multiply(0.09)) \
    .rename('Risk_Fuel_Plus')

risk_fuel_minus = fuel.multiply(0.32) \
    .add(dryness.multiply(0.34)) \
    .add(human.multiply(0.23)) \
    .add(slopeNorm.multiply(0.11)) \
    .rename('Risk_Fuel_Minus')

diffPlus = risk_fuel_plus.subtract(risk).abs().rename('Diff_Plus')
diffMinus = risk_fuel_minus.subtract(risk).abs().rename('Diff_Minus')

diffPlusStats = diffPlus.reduceRegion(
    reducer=ee.Reducer.mean().combine(ee.Reducer.max(), sharedInputs=True),
    geometry=roi.geometry(),
    scale=500,
    maxPixels=1e9
)

diffMinusStats = diffMinus.reduceRegion(
    reducer=ee.Reducer.mean().combine(ee.Reducer.max(), sharedInputs=True),
    geometry=roi.geometry(),
    scale=500,
    maxPixels=1e9
)

print("Risk mean/std:", stats.getInfo())
print("Validation summary:", validationStats.getInfo())
print("Sensitivity (+20% fuel):", diffPlusStats.getInfo())
print("Sensitivity (-20% fuel):", diffMinusStats.getInfo())

Risk mean/std: {'Risk_mean': 0.3109012817481249, 'Risk_stdDev': 0.04287088756703327}
Validation summary: {'agreement': 0, 'burned': 3095.227450980392, 'highRisk': 219}
Sensitivity (+20% fuel): {'Diff_Plus_max': 0.04709516285564752, 'Diff_Plus_mean': 0.031368236138445665}
Sensitivity (-20% fuel): {'Diff_Minus_max': 0.04709516285564752, 'Diff_Minus_mean': 0.03136823613844568}


In [11]:
Map = geemap.Map(center=[18.8, 98.9], zoom=8)

Map.addLayer(
    risk,
    {'min': 0.2, 'max': 0.8, 'palette': ['green', 'yellow', 'red']},
    'Wildfire Risk Index'
)

Map.addLayer(
    riskClass,
    {'min': 1, 'max': 3, 'palette': ['#27ae60', '#f1c40f', '#e74c3c']},
    'Wildfire Risk Class'
)

Map.addLayer(
    fireMask.selfMask(),
    {'palette': ['#8e44ad']},
    'Burned Area 2023'
)

Map.addLayer(
    diffPlus,
    {'min': 0, 'max': 0.15, 'palette': ['white', 'orange', 'red']},
    'Sensitivity Diff +20% Fuel'
)

Map.addLayer(
    diffMinus,
    {'min': 0, 'max': 0.15, 'palette': ['white', 'orange', 'red']},
    'Sensitivity Diff -20% Fuel'
)

Map.addLayerControl()
Map

Map(center=[18.8, 98.9], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(…